In [1]:
from pathlib import Path
import numpy as np

HAVE_MPL = True
try:
    import matplotlib.pyplot as plt
except ImportError as e:
    print(f"You likely need to install matplotlib: see original error message: {e}")
    HAVE_MPL = False

from forte2 import MCOptimizer, RHF, CISolver, State, System, write_orbital_cubes
from forte2.base_classes.params import DavidsonLiuParams

forte2: using 10 threads for parallel sections
[mods_manager] loading mod determinant_printing from /Users/fevange/.forte2/mods
[mods_manager] failed to load mod determinant_printing from /Users/fevange/.forte2/mods: partially initialized module 'forte2' from '/Users/fevange/Source/forte2-dev-2/forte2/__init__.py' has no attribute 'Determinant' (most likely due to a circular import)


In [19]:
def compute(r,C=None):
    xyz = f"""
    Li  0.0  0.0  0.0
    F   0.0  0.0  {r}
    """

    system = System(
        xyz=xyz,
        basis_set="cc-pVDZ",
        cholesky_tei=True,
        symmetry=False,
    )
    rhf = RHF(charge=0, e_tol=1.0e-8)(system)

    rhf.run()
    if C is not None:
        rhf.mos.C[0] = C

    singlet = State(system=system, multiplicity=1, ms=0.0)

    cas_solver = CISolver(
        states=singlet,
        core_orbitals=2,
        active_orbitals=8,
    )

    mc = MCOptimizer(cas_solver, e_tol=1.0e-12)(rhf)

    mc.run()

    return mc

In [21]:
mc0 = compute(1.57)

C0 = mc0.mos.C[0]
S0 = mc0.system.ints_overlap()

Point group symmetry detection not performed. Running in C1 symmetry.
Principal Atomic Positions (a.u.):
   LI   0.00000000   0.00000000   0.00000000
   F   0.00000000   0.00000000   2.96687002
Parsed 2 atoms with basis set of 28 functions.
  Max eigenvalue: 2.565e+00
  Min eigenvalue: 5.532e-02
  Condition number: 4.636e+01
  Inverse condition number: 2.157e-02
  Number of discarded eigenvalues: 0
  Number of kept eigenvalues: 28
  Largest discarded eigenvalue: 0.000e+00
  Smallest kept eigenvalue: 5.532e-02
Number of electrons: 12
Number of alpha electrons: 6
Number of beta electrons: 6
Ms: 0
Total charge: 0
Number of basis functions: 28
Number of orthogonalized basis functions: 28
Number of auxiliary basis functions: None
Energy convergence criterion: 1.000000e-08
Density convergence criterion: 1.000000e-06
DIIS acceleration: True

==> RHF SCF ROUTINE <==
Building B tensor using Cholesky decomposition
Temporary memory requirement for 4-index integrals: 0.00 GB
Memory requirements: 0

In [25]:
mc1 = compute(1.75)

Point group symmetry detection not performed. Running in C1 symmetry.
Principal Atomic Positions (a.u.):
   LI   0.00000000   0.00000000   0.00000000
   F   0.00000000   0.00000000   3.30702072
Parsed 2 atoms with basis set of 28 functions.
  Max eigenvalue: 2.538e+00
  Min eigenvalue: 6.198e-02
  Condition number: 4.095e+01
  Inverse condition number: 2.442e-02
  Number of discarded eigenvalues: 0
  Number of kept eigenvalues: 28
  Largest discarded eigenvalue: 0.000e+00
  Smallest kept eigenvalue: 6.198e-02
Number of electrons: 12
Number of alpha electrons: 6
Number of beta electrons: 6
Ms: 0
Total charge: 0
Number of basis functions: 28
Number of orthogonalized basis functions: 28
Number of auxiliary basis functions: None
Energy convergence criterion: 1.000000e-08
Density convergence criterion: 1.000000e-06
DIIS acceleration: True

==> RHF SCF ROUTINE <==
Building B tensor using Cholesky decomposition
Temporary memory requirement for 4-index integrals: 0.00 GB
Memory requirements: 0

In [26]:
import forte2
S1 = mc1.system.ints_overlap()
T = forte2.helpers.matrix_functions.invsqrt_matrix(C0.T @ S1 @ C0)[0]
C1_transported = C0 @ T

In [27]:
mc2 = compute(1.75, C=C1_transported)

Point group symmetry detection not performed. Running in C1 symmetry.
Principal Atomic Positions (a.u.):
   LI   0.00000000   0.00000000   0.00000000
   F   0.00000000   0.00000000   3.30702072
Parsed 2 atoms with basis set of 28 functions.
  Max eigenvalue: 2.538e+00
  Min eigenvalue: 6.198e-02
  Condition number: 4.095e+01
  Inverse condition number: 2.442e-02
  Number of discarded eigenvalues: 0
  Number of kept eigenvalues: 28
  Largest discarded eigenvalue: 0.000e+00
  Smallest kept eigenvalue: 6.198e-02
Number of electrons: 12
Number of alpha electrons: 6
Number of beta electrons: 6
Ms: 0
Total charge: 0
Number of basis functions: 28
Number of orthogonalized basis functions: 28
Number of auxiliary basis functions: None
Energy convergence criterion: 1.000000e-08
Density convergence criterion: 1.000000e-06
DIIS acceleration: True

==> RHF SCF ROUTINE <==
Building B tensor using Cholesky decomposition
Temporary memory requirement for 4-index integrals: 0.00 GB
Memory requirements: 0

In [15]:
with np.printoptions(precision=4, suppress=True):
    print(C0)
    print()
    print(C1_transported)

[[ 0.9943 -0.2081  0.0273 -0.      0.     -0.2003]
 [ 0.0119  0.4169 -1.1668  0.     -0.      0.3426]
 [ 0.     -0.      0.      0.9943 -0.1063 -0.    ]
 [-0.0235  0.3497 -0.7423 -0.      0.     -1.0298]
 [ 0.      0.     -0.     -0.1063 -0.9943 -0.    ]
 [ 0.0134  0.5461  1.355   0.     -0.      0.4909]]

[[ 0.9932 -0.1893  0.0899 -0.      0.     -0.1771]
 [-0.0008  0.53   -0.7232  0.     -0.      0.5083]
 [ 0.     -0.      0.      0.9943 -0.1063 -0.    ]
 [-0.0388  0.4703 -0.2512 -0.      0.     -0.8461]
 [ 0.     -0.      0.     -0.1063 -0.9943 -0.    ]
 [ 0.1129  0.6843  0.6879  0.     -0.      0.2189]]
